In [17]:
import requests
from PIL import Image
from io import BytesIO

# 1. 拿你提供的 2026-06-15 測試圖網址作為範例
target_url = "http://140.137.32.27/www/cwbrad/2026/06/15/20260615_1000.cwbrad.TV1.jpg"

print("正在從氣象署伺服器下載雷達原圖...")
response = requests.get(target_url)

if response.status_code == 200:
    # 2. 用 Pillow 讀取記憶體中的圖片數據
    img = Image.open(BytesIO(response.content))
    print(f"原圖載入成功，原始尺寸為: {img.size}") # 應該會顯示 (3600, 3600)

    # 3. 填入我們剛剛精算出來的兩階段幾何收網座標 (Left, Top, Right, Bottom)
    crop_box = (1200, 300, 3000, 1500)

    # 4. 進行物理盲切
    cropped_img = img.crop(crop_box)
    print(f"裁切完畢！新圖片尺寸為: {cropped_img.size}") # 完美對齊 (1800, 1200)

    # 5. 將原本很大的 PNG 轉換成 RGB，並以 60% 畫質儲存為 JPG (極致輕量化)
    # 如果是無地形白底圖，未來可以進一步在這邊做強制去背
    output_filename = "north_taiwan_radar_test.jpg"
    
    if cropped_img.mode in ('RGBA', 'P'):
        cropped_img = cropped_img.convert('RGB')
        
    cropped_img.save(output_filename, "JPEG", quality=60)
    print(f"🎉 測試圖優化存檔成功！檔名為: {output_filename}")
else:
    print(f"圖片下載失敗，錯誤碼: {response.status_code}")

正在從氣象署伺服器下載雷達原圖...
原圖載入成功，原始尺寸為: (1000, 1000)
裁切完畢！新圖片尺寸為: (1800, 1200)
🎉 測試圖優化存檔成功！檔名為: north_taiwan_radar_test.jpg


In [2]:
import os
import sys
import requests
from PIL import Image
from io import BytesIO
from datetime import datetime, timedelta

# ==========================================
# 接收主 Trigger 參數
# ==========================================
if len(sys.argv) > 3:
    Y, M, D = sys.argv[1], sys.argv[2], sys.argv[3]
else:
    Y, M, D = "2026", "06", "14"  # 手動測試預設值

OUTPUT_DIR = f"radar_archives/{Y}{M}{D}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"DATE: {Y}-{M}-{D} ...")
print("=" * 80)

# ==========================================
# 影像下載與核心函數
def download_and_crop(url, crop_box, output_path, label):
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            img = Image.open(BytesIO(response.content))
            cropped_img = img.crop(crop_box)
            if cropped_img.mode in ('RGBA', 'P'):
                cropped_img = cropped_img.convert('RGB')
            cropped_img.save(output_path, "JPEG", quality=90)
            return True
        else:
            return False
    except Exception as e:
        print(f"[{label}] 連線異常: {e}")
        return False

# ==========================================
# 任務 A & B 迴圈：台灣時間 10:00 - 18:00 (每 10 分鐘前進)
# ==========================================
print("正在處理：[a.雷達回波] 與 [b.逐時閃電] (每 10 分鐘)...")
dt_10_18 = datetime(int(Y), int(M), int(D), 10, 0, 0)
end_10_18 = datetime(int(Y), int(M), int(D), 18, 0, 0)

while dt_10_18 <= end_10_18:
    # 同時取得小時與分鐘，例如 "1000", "1010", "1020" ...
    hhmm = dt_10_18.strftime("%H%M")
    hh_mm_display = dt_10_18.strftime("%H:%M") # 用於 Log 顯示
    
    # ➔ a. 雷達回波無地形 
    url_a = f"http://140.137.32.27/www/cwbrad/{Y}/{M}/{D}/{Y}{M}{D}_{hhmm}.cwbrad.TV1.jpg"
    path_a = os.path.join(OUTPUT_DIR, f"radar_north_{Y}{M}{D}_{hhmm}.jpg")
    box_a = (333, 83, 833, 417)
    
    if download_and_crop(url_a, box_a, path_a, "雷達"):
        print(f"雷達圖 {hh_mm_display} 裁切成功 ➔ {os.path.basename(path_a)}")
        
    # ➔ b. 逐時閃電資料
    url_b = f"http://140.137.32.27/www/cwblgt/{Y}/{M}/{D}/{Y}{M}{D}_{hhmm}.cwblgt.lgtx.gif"
    path_b = os.path.join(OUTPUT_DIR, f"radar_lgt_north_{Y}{M}{D}_{hhmm}.jpg")
    box_b = (368, 0, 1110, 492) 
    
    if download_and_crop(url_b, box_b, path_b, "閃電"):
        print(f"閃電圖 {hh_mm_display} 裁切成功 ➔ {os.path.basename(path_b)}")
        
    # 調整為每次前進 10 分鐘
    dt_10_18 += timedelta(minutes=10)

print("-" * 80)

# ==========================================
# 任務 C 迴圈：台灣時間 09:00 - 15:00 (每半小時前進)
# ==========================================
OUTPUT_DIR2 = f"sfc_archives/{Y}{M}{D}"
os.makedirs(OUTPUT_DIR2, exist_ok=True)
print("🌡️ 正在處理：[c.時雨量+風] 與 [溫度分布+風] (半小時步進)...")
dt_09_15 = datetime(int(Y), int(M), int(D), 9, 0, 0)
end_09_15 = datetime(int(Y), int(M), int(D), 15, 0, 0)

while dt_09_15 <= end_09_15:
    # 格式化為小時與分鐘 (例如: 0900, 0930)
    hhmm = dt_09_15.strftime("%H%M")
    hh_mm_log = dt_09_15.strftime("%H:%M") # 用於 Log 顯示
    
    # ➔ c-1. 時雨量+風
    url_c1 = f"http://140.137.32.27/www/rainhr/{Y}/{M}/{D}/{Y}{M}{D}_{hhmm}.cwbrain.rainhrw.jpg"
    path_c1 = os.path.join(OUTPUT_DIR2, f"rain_wind_north_{Y}{M}{D}_{hhmm}.jpg")
    box_c1 = (150, 0, 880, 340)
    
    if download_and_crop(url_c1, box_c1, path_c1, "時雨量+風"):
        print(f"雨量風場 {hh_mm_log} 裁切成功 ➔ {os.path.basename(path_c1)}")
        
    # ➔ c-2. 溫度分布+風
    url_c2 = f"http://140.137.32.27/www/cwbgtp/{Y}/{M}/{D}/{Y}{M}{D}_{hhmm}.cwbtemp.GTPw.jpg"
    path_c2 = os.path.join(OUTPUT_DIR2, f"temp_wind_north_{Y}{M}{D}_{hhmm}.jpg")
    box_c2 = (420, 0, 1500, 600)
    
    if download_and_crop(url_c2, box_c2, path_c2, "溫度+風"):
        print(f"溫度風場 {hh_mm_log} 裁切成功 ➔ {os.path.basename(path_c2)}")
        
    # 每次前進 30 分鐘
    dt_09_15 += timedelta(minutes=30)

print("=" * 80)
print(f"DONE: {Y}-{M}-{D}")

DATE: 2026-06-14 ...
🌡️ 正在處理：[c.時雨量+風] 與 [溫度分布+風] (半小時步進)...
雨量風場 09:00 裁切成功 ➔ rain_wind_north_20260614_0900.jpg
溫度風場 09:00 裁切成功 ➔ temp_wind_north_20260614_0900.jpg
雨量風場 09:30 裁切成功 ➔ rain_wind_north_20260614_0930.jpg
雨量風場 10:00 裁切成功 ➔ rain_wind_north_20260614_1000.jpg
溫度風場 10:00 裁切成功 ➔ temp_wind_north_20260614_1000.jpg
雨量風場 10:30 裁切成功 ➔ rain_wind_north_20260614_1030.jpg
雨量風場 11:00 裁切成功 ➔ rain_wind_north_20260614_1100.jpg
溫度風場 11:00 裁切成功 ➔ temp_wind_north_20260614_1100.jpg
雨量風場 11:30 裁切成功 ➔ rain_wind_north_20260614_1130.jpg
雨量風場 12:00 裁切成功 ➔ rain_wind_north_20260614_1200.jpg
溫度風場 12:00 裁切成功 ➔ temp_wind_north_20260614_1200.jpg
雨量風場 12:30 裁切成功 ➔ rain_wind_north_20260614_1230.jpg
雨量風場 13:00 裁切成功 ➔ rain_wind_north_20260614_1300.jpg
溫度風場 13:00 裁切成功 ➔ temp_wind_north_20260614_1300.jpg
雨量風場 13:30 裁切成功 ➔ rain_wind_north_20260614_1330.jpg
雨量風場 14:00 裁切成功 ➔ rain_wind_north_20260614_1400.jpg
溫度風場 14:00 裁切成功 ➔ temp_wind_north_20260614_1400.jpg
雨量風場 14:30 裁切成功 ➔ rain_wind_north_20260614_1430.jpg
雨量風

In [19]:
import os
import sys
import requests
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta

# ==========================================
# 📥 1. 接收主 Trigger 參數
# ==========================================
if len(sys.argv) > 3:
    Y, M, D = sys.argv[1], sys.argv[2], sys.argv[3]
else:
    Y, M, D = "2026", "06", "14"

API_TOKEN = "CWB-B1034BF8-0855-45E6-8F10-7C7D4DB194AA"
OUTPUT_DIR = f"stn_archives/{Y}{M}{D}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f" DATE: ({Y}-{M}-{D})...")
print("=" * 80)

raw_data_records = []

# ==========================================
# 逐時 10:00 - 18:00 氣象站 XML
# ==========================================
dt_10_18 = datetime(int(Y), int(M), int(D), 10, 0, 0)
end_10_18 = datetime(int(Y), int(M), int(D), 18, 0, 0)

# 依據資料範例，定義絕對正確的命名空間首碼
NS = "{urn:cwa:gov:tw:cwacommon:0.1}"

while dt_10_18 <= end_10_18:
    hh = dt_10_18.strftime("%H")
    
    url = f"https://opendata.cwa.gov.tw/historyapi/v1/getData/O-A0001-001/{Y}/{M}/{D}/{hh}/00/00?Authorization={API_TOKEN}&format=XML"
    
    try:
        response = requests.get(url, timeout=15)
        if response.status_code == 200:
            # 解析 XML
            root = ET.fromstring(response.content)
            
            # 1. 抓出所有的 Station 節點
            stations = root.findall(f".//{NS}Station")
            
            for st in stations:
                # 2. 縣市名稱 (GeoInfo -> CountyName)
                county_node = st.find(f"./{NS}GeoInfo/{NS}CountyName")
                county = county_node.text if county_node is not None else ""
                
                # ➔ 進行雙北地理篩選
                if county in ["臺北市", "新北市"]:
                    st_name_node = st.find(f"./{NS}StationName")
                    st_id_node = st.find(f"./{NS}StationId")
                    
                    st_name = st_name_node.text if st_name_node is not None else "未知"
                    st_id = st_id_node.text if st_id_node is not None else ""
                    
                    # 3. 每小時累積雨量 (WeatherElement -> Now -> Precipitation)
                    rain_node = st.find(f"./{NS}WeatherElement/{NS}Now/{NS}Precipitation")
                    try:
                        rain = float(rain_node.text) if rain_node is not None else 0.0
                        rain = rain if rain >= 0 else 0.0
                    except: rain = 0.0
                    
                    # 4. 即時氣溫 (WeatherElement -> AirTemperature)
                    temp_node = st.find(f"./{NS}WeatherElement/{NS}AirTemperature")
                    try:
                        temp = float(temp_node.text) if temp_node is not None else None
                        temp = temp if (temp and temp > -50) else None
                    except: temp = None
                    
                    # 5. 風速與風向 (WeatherElement -> WindSpeed / WindDirection)
                    speed_node = st.find(f"./{NS}WeatherElement/{NS}WindSpeed")
                    dir_node = st.find(f"./{NS}WeatherElement/{NS}WindDirection")
                    try: w_speed = float(speed_node.text) if speed_node is not None else 0.0
                    except: w_speed = 0.0
                    try: w_dir = float(dir_node.text) if dir_node is not None else 0.0
                    except: w_dir = 0.0
                    
                    raw_data_records.append({
                        "StationName": st_name,
                        "StationId": st_id,
                        "Hour": f"{hh}:00",
                        "Rain": rain,
                        "Temp": temp,
                        "WindSpeed": w_speed,
                        "WindDir": w_dir
                    })
            print(f"SUCCESS {hh}:00 DATA")
        else:
            print(f"  ⚪ {hh}:00 NO RESPONSE (HTTP {response.status_code})")
    except Exception as e:
        print(f"  ❌ {hh}:00 ERROR: {e}")
        
    dt_10_18 += timedelta(minutes=60)

# ==========================================
if not raw_data_records:
    print("NO DATA")
    sys.exit()

df = pd.DataFrame(raw_data_records)
df = df.replace({col: {np.nan: None} for col in df.columns})

# 總累積雨量 Top 10
df_rain = df.groupby(["StationName", "StationId"])["Rain"].sum().reset_index()
top10_rain = df_rain.sort_values(by="Rain", ascending=False).head(10).to_dict(orient="records")

# 當日最高氣溫 Top 10
df_temp = df.groupby(["StationName", "StationId"])["Temp"].max().reset_index()
top10_temp = df_temp.sort_values(by="Temp", ascending=False).head(10).to_dict(orient="records")

# 逐時風向風速矩陣
hourly_wind_matrix = df.to_dict(orient="records")

# ==========================================
# 為 index.html 專用 JSON 檔案
final_summary_json = {
    "meta": {
        "target_date": f"{Y}-{M}-{D}",
        "calculated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    },
    "top10_precipitation": top10_rain,
    "top10_max_temperature": top10_temp,
    "hourly_wind_data": hourly_wind_matrix
}

import json
json_output_path = os.path.join(OUTPUT_DIR, "stations_summary.json")
with open(json_output_path, "w", encoding="utf-8") as f:
    json.dump(final_summary_json, f, ensure_ascii=False, indent=2)

print("-" * 80)
print(f" DONE ➔ {json_output_path}")
print("=" * 80)

 DATE: (2026-06-14)...
SUCCESS 10:00 DATA
SUCCESS 11:00 DATA
SUCCESS 12:00 DATA
SUCCESS 13:00 DATA
SUCCESS 14:00 DATA
SUCCESS 15:00 DATA
SUCCESS 16:00 DATA
SUCCESS 17:00 DATA
SUCCESS 18:00 DATA
--------------------------------------------------------------------------------
 DONE ➔ stn_archives/20260614\stations_summary.json


In [ ]:
import os
import sys
import requests

# ==========================================
# 接收主 Trigger 參數
if len(sys.argv) > 3:
    Y, M, D = sys.argv[1], sys.argv[2], sys.argv[3]
else:
    Y, M, D = "2026", "06", "14"  # 地端測試預設值

OUTPUT_DIR = f"cwaFig_archives/{Y}{M}{D}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 格式化各式網址所需的日期字串
YYMMDD = f"{Y[2:]}{M}{D}"  
YYYYMMDD = f"{Y}{M}{D}"  

print(f"DATE: {Y}-{M}-{D} 00UTC (08:00 LST) ...")
print("=" * 80)

# ==========================================
def download_static_map(primary_url, fallback_url, output_name, label):
    out_path = os.path.join(OUTPUT_DIR, output_name)
    
    # 嘗試優先方案
    try:
        res = requests.get(primary_url, timeout=10)
        if res.status_code == 200:
            with open(out_path, "wb") as f:
                f.write(res.content)
            print(f"primary_url success ➔ {output_name}")
            return
    except Exception:
        pass
        
    # 優先方案失敗，啟動備援方案
    try:
        res = requests.get(fallback_url, timeout=10)
        if res.status_code == 200:
            with open(out_path, "wb") as f:
                f.write(res.content)
            print(f"fallback_url success ➔ {output_name}")
    except Exception as e:
        print(f"Download failed: {e}")

# ==========================================
# ➔ g-1. 台北 46692 探空圖
url_skewt_46692_pri = f"https://npd1.cwa.gov.tw/NPD/irisme_data/Weather/SKEWT/SKW___000_{YYMMDD}00_46692.gif"
url_skewt_46692_fal = f"http://140.137.32.27/www/skw2/{Y}/{M}/{D}/{YYYYMMDD}_0000.46692.skw.jpg"
download_static_map(url_skewt_46692_pri, url_skewt_46692_fal, "skewt_taipei.gif", "台北探空")

# ➔ g-2. 彭佳嶼 46695 探空圖
url_skewt_46695_pri = f"https://npd1.cwa.gov.tw/NPD/irisme_data/Weather/SKEWT/SKW___000_{YYMMDD}00_46695.gif"
url_skewt_46695_fal = f"http://140.137.32.27/www/skw2/{Y}/{M}/{D}/{YYYYMMDD}_0000.46695.skw.jpg"
download_static_map(url_skewt_46695_pri, url_skewt_46695_fal, "skewt_pengjia.gif", "彭佳嶼探空")

# ➔ h-1. 地面天氣圖 
url_sfc_pri = f"http://140.137.32.27/www/cwbmap/{Y}/{M}/{D}/{YYYYMMDD}_0000.cwbmap.sfcmap.gif"
url_sfc_fal = f"https://npd1.cwa.gov.tw/NPD/irisme_data/Weather/ANALYSIS/GRA___000_{YYMMDD}00_103.gif"
download_static_map(url_sfc_pri, url_sfc_fal, "synoptic_sfc.png", "地面天氣圖")

# ➔ h-2. 850 hPa 高空分析圖
url_850_pri = f"https://npd1.cwa.gov.tw/NPD/irisme_data/Weather/HLANALYSIS/GRA___000_{YYMMDD}00_001.gif"
url_850_fal = f"http://140.137.32.27/www/cwbmap/{Y}/{M}/{D}/{YYYYMMDD}_0000.cwbmap.asu850.gif"
download_static_map(url_850_pri, url_850_fal, "synoptic_850.gif", "850hPa分析圖")

# ➔ h-3. 500 hPa 高空分析圖
url_500_pri = f"https://npd1.cwa.gov.tw/NPD/irisme_data/Weather/HLANALYSIS/GRA___000_{YYMMDD}00_003.gif"
url_500_fal = f"http://140.137.32.27/www/cwbmap/{Y}/{M}/{D}/{YYYYMMDD}_0000.cwbmap.asu500.gif"
download_static_map(url_500_pri, url_500_fal, "synoptic_500.gif", "500hPa分析圖")

print("=" * 80)
print(f"DONE! ")

DATE: 2026-06-15 00UTC (08:00 LST) ...
primary_url success ➔ skewt_taipei.gif
primary_url success ➔ skewt_pengjia.gif
primary_url success ➔ synoptic_sfc.png
primary_url success ➔ synoptic_850.gif
primary_url success ➔ synoptic_500.gif
DONE! 
